# Thử nghiệm Silero VAD (GPU CUDA)

Phát hiện **đoạn có giọng nói**, xem waveform, nghe lại và xuất WAV/JSON. VAD không chuyển giọng nói thành văn bản.

**Cách chạy:** chọn kernel Python 3.12 trên máy có NVIDIA GPU → chạy cell cài đặt → sửa `AUDIO_PATH` → Run All.
Trên Colab: chọn **Runtime → Change runtime type → T4 GPU** trước khi chạy.
Mặc định dùng `ttshub-ghep-im-lang-3s.mp3`: đoạn `b6c9e15d` → thêm 3 giây im lặng → đoạn `7e6fdfaa`. Tìm file ở `/content` hoặc trong repo. Nếu chạy trên Colab, upload file vào `/content` rồi đặt đường dẫn tương ứng.

Dùng `silero-vad==6.2.0` giống `Sub-ASR/pyproject.toml`. Chạy bằng PyTorch CUDA 12.8; model và tensor audio được chuyển sang GPU. Không cần model ASR hay API key. Nếu CUDA không khả dụng, notebook dừng và hướng dẫn kiểm tra kernel/driver.
Tham khảo [Silero VAD chính thức](https://github.com/snakers4/silero-vad).
Bản PyTorch CUDA theo [hướng dẫn cài đặt chính thức](https://pytorch.org/get-started/previous-versions/#v280).
Đọc audio bằng [FFmpeg](https://ffmpeg.org/ffmpeg.html) trực tiếp vào RAM. Cell xuất cuối notebook vẫn lưu WAV giọng nói và JSON nếu bạn chạy cell đó; đây là kết quả VAD, không phải file giải mã trung gian.

In [ ]:
# Cài bản CUDA kể cả khi kernel đang có torch CPU cùng phiên bản.
# Sau khi cài, Restart Kernel/Restart Session rồi chạy lại từ cell import.
%pip install --upgrade torch==2.8.0+cu128 torchaudio==2.8.0+cu128 --index-url https://download.pytorch.org/whl/cu128
%pip install silero-vad==6.2.0 soundfile imageio-ffmpeg matplotlib ipykernel

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path
from time import perf_counter
from importlib.metadata import version

import numpy as np
import soundfile as sf
import torch
import matplotlib.pyplot as plt
import imageio_ffmpeg
from IPython.display import Audio, display
from silero_vad import load_silero_vad, get_speech_timestamps

torch.set_num_threads(1)
if not torch.cuda.is_available():
    raise RuntimeError(
        f'CUDA không khả dụng (torch={torch.__version__}, CUDA build={torch.version.cuda}). '
        'Colab: bật T4 GPU. Máy local: kiểm tra NVIDIA driver và kernel Python; '
        'chạy cell cài CUDA, sau đó restart kernel.'
    )
device = torch.device('cuda:0')
model = load_silero_vad(onnx=False).to(device).eval()
assert next(model.parameters()).is_cuda
print(f"torch={torch.__version__}, silero-vad={version('silero-vad')}, device={device}")
print(f'GPU: {torch.cuda.get_device_name(device)} | CUDA build: {torch.version.cuda}')

## Chọn audio và tham số

Đường dẫn tương đối tính từ thư mục làm việc của kernel. Chọn MP3, WAV, FLAC hoặc định dạng FFmpeg hỗ trợ. FFmpeg giải mã trên CPU thành float32 mono 16 kHz trong RAM; Silero suy luận trên GPU. Không tạo WAV trung gian.
Nếu file hỏng một phần, notebook hiển thị thông báo FFmpeg; chỉ tiếp tục khi giải mã có dữ liệu và thành công. Nếu đọc file thất bại, dừng và xử lý lỗi trước khi chạy các cell sau.
- `THRESHOLD`: tăng để phát hiện chặt hơn, giảm để nhạy hơn (có thể nhận nhầm tiếng ồn).
- `MIN_SPEECH_MS`: loại đoạn giọng nói quá ngắn.
- `MIN_SILENCE_MS`: khoảng lặng cần để tách hai đoạn.
- `SPEECH_PAD_MS`: giữ thêm hai đầu để hạn chế cắt mất âm.

Sau khi chỉnh tham số, chạy lại từ cell phát hiện. Sau khi đổi file, chạy lại từ cell đọc audio.

In [ ]:
# File thử: hai đoạn nói, thêm 3 giây im lặng ở giữa.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'Silero_VAD_Test.ipynb').is_file()), Path.cwd())
TEST_AUDIO_NAME = 'ttshub-ghep-im-lang-3s.mp3'
AUDIO_PATH = (Path('/content') / TEST_AUDIO_NAME
              if (Path('/content') / TEST_AUDIO_NAME).is_file() else ROOT / TEST_AUDIO_NAME)
# AUDIO_PATH = Path('/content/my_audio.mp3')

SAMPLE_RATE = 16000
THRESHOLD = 0.5
MIN_SPEECH_MS = 250
MIN_SILENCE_MS = 200
SPEECH_PAD_MS = 100
OUTPUT_DIR = ROOT / 'silero_vad_outputs'

In [ ]:
def decode_audio_memory(path, sample_rate):
    """FFmpeg → PCM float32 mono trong RAM; không ghi file trung gian."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Không tìm thấy {path}. Sửa AUDIO_PATH ở cell trên.')
    ffmpeg = shutil.which('ffmpeg') or imageio_ffmpeg.get_ffmpeg_exe()
    result = subprocess.run(
        [ffmpeg, '-nostdin', '-hide_banner', '-loglevel', 'warning',
         '-i', str(path.resolve()), '-map', '0:a:0', '-ac', '1', '-ar', str(sample_rate),
         '-c:a', 'pcm_f32le', '-f', 'f32le', 'pipe:1'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )
    diagnostic = result.stderr.decode('utf-8', errors='replace').strip()
    if result.returncode != 0 or not result.stdout:
        raise RuntimeError(f'FFmpeg không giải mã được {path.name}:\n{diagnostic}')
    samples = np.frombuffer(result.stdout, dtype='<f4').astype(np.float32, copy=True)
    if not samples.size or not np.isfinite(samples).all():
        raise ValueError('Audio giải mã rỗng hoặc chứa NaN/Inf.')
    return samples, diagnostic


# Xóa dữ liệu lần trước để cell sau không dùng nhầm audio cũ nếu đọc file thất bại.
audio = wave = timestamps = segments = speech_wave = duration = None
wave, decode_diagnostic = decode_audio_memory(AUDIO_PATH, SAMPLE_RATE)
if decode_diagnostic:
    print('Thông báo FFmpeg (audio có thể đã bỏ qua phần dữ liệu hỏng):')
    print(decode_diagnostic)
audio = torch.from_numpy(wave).to(device)
assert audio.is_cuda and next(model.parameters()).is_cuda
duration = len(wave) / SAMPLE_RATE
print(f'{AUDIO_PATH.name} → mono {SAMPLE_RATE} Hz, {duration:.2f} giây (giải mã trong RAM)')
display(Audio(wave, rate=SAMPLE_RATE, normalize=False))

## Phát hiện giọng nói

Giữ timestamp nội bộ theo sample để cắt audio chính xác; bảng/JSON chuyển sang giây.
Thời gian đo chỉ gồm VAD sau một lần warm-up, không gồm đọc file hoặc tải model. `RTF < 1` nghĩa là xử lý nhanh hơn thời lượng audio.
Đồng bộ CUDA trước/sau phép đo để tính cả thời gian GPU hoàn tất. Đọc file, resample và chuyển audio lên GPU nằm ngoài phép đo.
Model suy luận trên GPU; tách đoạn theo xác suất chạy bằng Python. Với từng chunk ngắn, GPU có thể chậm hơn CPU vì overhead gọi kernel và đồng bộ.

In [ ]:
assert 0 < THRESHOLD < 1
assert MIN_SPEECH_MS >= 0 and MIN_SILENCE_MS >= 0 and SPEECH_PAD_MS >= 0

with torch.inference_mode():
    get_speech_timestamps(torch.zeros(SAMPLE_RATE, device=device), model, sampling_rate=SAMPLE_RATE)
    torch.cuda.synchronize(device)
    started = perf_counter()
    timestamps = get_speech_timestamps(
        audio, model, sampling_rate=SAMPLE_RATE,
        threshold=THRESHOLD,
        min_speech_duration_ms=MIN_SPEECH_MS,
        min_silence_duration_ms=MIN_SILENCE_MS,
        speech_pad_ms=SPEECH_PAD_MS,
        return_seconds=False,
    )
    torch.cuda.synchronize(device)
    elapsed = perf_counter() - started

segments = [dict(start=t['start'] / SAMPLE_RATE, end=t['end'] / SAMPLE_RATE,
                 duration=(t['end'] - t['start']) / SAMPLE_RATE) for t in timestamps]
speech_seconds = sum(s['duration'] for s in segments)
print(f'{len(segments)} đoạn | Giọng nói (gồm padding): {speech_seconds:.2f}/{duration:.2f}s '
      f'({100 * speech_seconds / duration:.1f}%)')
print(f'VAD: {elapsed * 1000:.1f} ms | RTF: {elapsed / duration:.4f}')
print(f"{'#':>3} {'Bắt đầu (s)':>12} {'Kết thúc (s)':>13} {'Dài (s)':>10}")
for i, s in enumerate(segments, 1):
    print(f"{i:>3} {s['start']:>12.3f} {s['end']:>13.3f} {s['duration']:>10.3f}")
if not segments:
    print('Không phát hiện giọng nói. Thử giảm THRESHOLD hoặc chọn audio khác.')

## Kiểm tra file ghép có khoảng nghỉ 3 giây

Chạy các cell từ cấu hình → đọc audio → VAD → cell bên dưới. Nghe audio gốc để giữ nguyên khoảng nghỉ; audio ghép sau VAD sẽ bỏ các khoảng lặng.
Với `MIN_SILENCE_MS=200`, khoảng nghỉ 3 giây đủ dài để tách đoạn nếu Silero nhận ra giọng nói ở cả hai phía. Một câu vẫn có thể bị tách thành nhiều đoạn.
Khoảng lặng đo giữa các vùng VAD có thể khác 3 giây vì file nguồn có khoảng lặng và VAD thêm padding. Notebook này kiểm tra phát hiện tiếng nói; chưa chạy ASR/LLM hoặc phân loại bổ sung/thay thế yêu cầu.
Trên Colab, upload `ttshub-ghep-im-lang-3s.mp3` vào `/content` trước khi chạy.


In [ ]:
print(f'File đang thử: {AUDIO_PATH.name} | {duration:.3f} giây | {len(segments)} vùng giọng nói')
vad_gaps = []
for previous, following in zip(segments, segments[1:]):
    gap = following['start'] - previous['end']
    vad_gaps.append(gap)
    print(f"Khoảng nghỉ: {previous['end']:.3f} → {following['start']:.3f}s, dài {gap:.3f}s")
if vad_gaps:
    print(f'Khoảng nghỉ dài nhất giữa các vùng VAD: {max(vad_gaps):.3f} giây')
else:
    print('Chưa có hai vùng giọng nói để đo khoảng nghỉ. Xem waveform và nghe lại audio gốc.')


In [ ]:
fig, ax = plt.subplots(figsize=(15, 3))
ax.plot(np.arange(len(wave)) / SAMPLE_RATE, wave, linewidth=0.5, color='steelblue')
for i, s in enumerate(segments):
    ax.axvspan(s['start'], s['end'], color='limegreen', alpha=0.3,
               label='Giọng nói' if i == 0 else None)
ax.set(xlabel='Thời gian (giây)', ylabel='Biên độ', title=AUDIO_PATH.name, xlim=(0, duration))
if segments:
    ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# Ghép các đoạn có giọng nói; khoảng lặng giữa các đoạn sẽ bị bỏ.
speech_wave = (np.concatenate([wave[t['start']:t['end']] for t in timestamps])
               if timestamps else np.empty(0, dtype=np.float32))
if speech_wave.size:
    print('Audio sau VAD:')
    display(Audio(speech_wave, rate=SAMPLE_RATE, normalize=False))
    for i, t in enumerate(timestamps[::], 1):
        print(f"Đoạn {i}: {t['start']/SAMPLE_RATE:.3f}–{t['end']/SAMPLE_RATE:.3f}s")
        display(Audio(wave[t['start']:t['end']], rate=SAMPLE_RATE, normalize=False))
else:
    print('Không có đoạn giọng nói để nghe hoặc lưu WAV.')

In [ ]:
# Kiểm tra kết quả và trường hợp im lặng bằng model thật.
assert audio.is_cuda and next(model.parameters()).is_cuda
previous_end = 0
for t in timestamps:
    assert previous_end <= t['start'] < t['end'] <= len(wave)
    previous_end = t['end']
assert speech_wave.size == sum(t['end'] - t['start'] for t in timestamps)
with torch.inference_mode():
    silence_segments = get_speech_timestamps(
        torch.zeros(2 * SAMPLE_RATE, device=device), model, sampling_rate=SAMPLE_RATE,
        threshold=THRESHOLD, min_speech_duration_ms=MIN_SPEECH_MS,
        min_silence_duration_ms=MIN_SILENCE_MS, speech_pad_ms=SPEECH_PAD_MS,
    )
assert silence_segments == [], f'Model nhận giọng nói trên im lặng: {silence_segments}'
print('PASS: model/audio trên CUDA, timestamp hợp lệ, độ dài audio ghép đúng, im lặng không có giọng nói.')

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_path = OUTPUT_DIR / f'{AUDIO_PATH.stem}_vad.json'
report = dict(
    input=str(AUDIO_PATH.resolve()), silero_vad_version=version('silero-vad'),
    device=str(device), gpu=torch.cuda.get_device_name(device), cuda_build=torch.version.cuda,
    sample_rate=SAMPLE_RATE, audio_seconds=duration, speech_seconds=speech_seconds,
    vad_seconds=elapsed, rtf=elapsed / duration,
    parameters=dict(threshold=THRESHOLD, min_speech_duration_ms=MIN_SPEECH_MS,
                    min_silence_duration_ms=MIN_SILENCE_MS, speech_pad_ms=SPEECH_PAD_MS),
    decoder="ffmpeg", decode_diagnostic=decode_diagnostic,
    segments=segments,
)
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu:', report_path)
if speech_wave.size:
    speech_path = OUTPUT_DIR / f'{AUDIO_PATH.stem}_speech.wav'
    sf.write(str(speech_path), speech_wave, SAMPLE_RATE, subtype='PCM_16')
    print('Đã lưu:', speech_path)

## Thử với tiếng ồn

Đổi `AUDIO_PATH` thành một file cùng thư mục, rồi chạy lại từ cell đọc audio:
- `vi_017_doorslam_events_snr00.wav`: giọng nói kèm tiếng đóng cửa.
- `keys_events_only_00.wav`: chỉ tiếng chìa khóa, để quan sát phát hiện nhầm.

Nghe và xem waveform khi thay `THRESHOLD` (ví dụ 0.3, 0.5, 0.7). Không có nhãn timestamp chuẩn trong notebook này, nên số đoạn và tỉ lệ giọng nói **không phải** chỉ số accuracy.
File xuất cùng tên sẽ được ghi đè khi chạy lại cell lưu kết quả.